# Model bake-off: skore + MLflow

This notebook mirrors `examples/01_bakeoff.py` as an interactive walkthrough. We run 5-fold cross-validation on the Digits dataset for three scikit-learn classifiers, log each one as its own MLflow run through `skore.Project`, and then rank them to surface the stability-vs-mean tradeoff — the highest-mean model is not always the most stable across folds. Keep the markdown cells as a reading guide; every code cell below has its own output so you can see the result of each step without re-running the notebook.

In [1]:
import os

from sklearn.datasets import load_digits
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from skore import Project, compare, evaluate

# Configuration — override via environment variables or edit directly
PROJECT = os.environ.get("PROJECT", "bakeoff-01")
TRACKING_URI = os.environ.get("TRACKING_URI", "sqlite:///mlflow.db")

# Metric used for the ranking punchline (must appear in the summarize() output)
METRIC = "Accuracy"

## Dataset: Digits

We use `load_digits` rather than the classic Iris set because Digits has enough samples (1,797) and features (64 pixel intensities) to actually separate the three estimators, which makes the stability-vs-mean tradeoff visible instead of noise.

In [2]:
X, y = load_digits(return_X_y=True, as_frame=True)
print("X.shape:", X.shape)

X.shape: (1797, 64)


## Three reasonable baselines

- **`hgb` — `HistGradientBoostingClassifier`**: a modern gradient-boosted tree ensemble, usually a strong default on tabular data with no scaling required.
- **`logreg` — `StandardScaler` + `LogisticRegression`**: the scaled linear baseline; cheap to fit and a good sanity check that the problem needs non-linearity.
- **`rf` — `RandomForestClassifier`**: bagged trees, a complementary ensemble that trades bias for variance differently than boosting.

All three use `random_state=0` so the per-fold scores are reproducible.

In [3]:
estimators = {
    "hgb": HistGradientBoostingClassifier(random_state=0),
    "logreg": Pipeline(
        [
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=0)),
        ]
    ),
    "rf": RandomForestClassifier(random_state=0),
}
estimators

{'hgb': HistGradientBoostingClassifier(random_state=0),
 'logreg': Pipeline(steps=[('scaler', StandardScaler()),
                 ('clf', LogisticRegression(max_iter=1000, random_state=0))]),
 'rf': RandomForestClassifier(random_state=0)}

## One MLflow run per estimator

Opening a `Project` with `mode="mlflow"` turns the given `tracking_uri` into an MLflow experiment named after `PROJECT`. Every call to `project.put(key, report)` creates a dedicated MLflow run that stores the serialized `CrossValidationReport` alongside the per-fold and aggregated metrics, so each estimator shows up as its own row in the MLflow UI.

In [4]:
project = Project(PROJECT, mode="mlflow", tracking_uri=TRACKING_URI)

reports = {}
for slug, estimator in estimators.items():
    print(f"Evaluating {slug}...")
    report = evaluate(estimator, X, y, splitter=5)
    project.put(slug, report)
    reports[slug] = report

list(reports)

2026/04/19 10:45:28 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/04/19 10:45:28 INFO mlflow.store.db.utils: Updating database tables


2026/04/19 10:45:28 INFO mlflow.tracking.fluent: Experiment with name 'bakeoff-01' does not exist. Creating a new experiment.


Output()

Evaluating hgb...


...

2026/04/19 10:46:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/04/19 10:46:27 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


2026/04/19 10:46:29 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


2026/04/19 10:46:30 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


['hgb', 'logreg', 'rf']

## Comparing reports side-by-side

`compare(reports)` returns a `ComparisonReport` that aligns the three `CrossValidationReport`s on the same fold structure. Unlike a hand-rolled table of means, the comparison exposes the full per-fold distribution for every metric and handles aggregation so the resulting DataFrame (via `.metrics.summarize().frame()`) already carries means, standard deviations, and the metric names you would expect.

In [5]:
comparison = compare(reports)
comparison.metrics.summarize().frame()

Output()

mean                           std  \
Estimator                              hgb    logreg        rf       hgb   
Metric           Label / Average                                           
Accuracy                          0.934895  0.920449  0.937699  0.018513   
Precision        0                0.978078  1.000000  0.983325  0.030023   
                 1                0.934168  0.852048  0.945455  0.029928   
                 2                0.966967  0.967385  0.961924  0.030161   
                 3                0.960044  0.957646  0.954474  0.036727   
                 4                0.961965  0.966890  0.971905  0.020675   
                 5                0.927310  0.951029  0.952706  0.037277   
                 6                0.967822  0.947605  0.972806  0.029415   
                 7                0.939889  0.941941  0.928649  0.039950   
                 8                0.905411  0.856301  0.866797  0.068385   
                 9                0.846837  0.832298  0.886395  0.098850   
Recall           0                0.988730  0.977302  0.988730  0.015434   
                 1                0.928829  0.901351  0.950450  0.035763   
                 2                0.943333  0.927460  0.915714  0.049607   
                 3                0.879580  0.868468  0.895946  0.140679   
                 4                0.950000  0.944745  0.966667  0.036218   
                 5                0.945195  0.928979  0.945345  0.051277   
                 6                0.972222  0.966667  0.977778  0.019642   
                 7                0.927778  0.911111  0.944444  0.103190   
                 8                0.884538  0.878655  0.902017  0.074600   
                 9                0.927778  0.900000  0.888889  0.075051   
ROC AUC          0                0.999965  0.999965  0.999815  0.000079   
                 1                0.997504  0.992681  0.997620  0.001896   
                 2                0.999457  0.996977  0.999256  0.000592   
                 3                0.995010  0.995333  0.996718  0.006653   
                 4                0.997093  0.996300  0.997179  0.005740   
                 5                0.997410  0.997627  0.997168  0.001548   
                 6                0.998148  0.997973  0.998962  0.003860   
                 7                0.998488  0.997992  0.998234  0.001635   
                 8                0.995911  0.991434  0.995970  0.002852   
                 9                0.995877  0.993594  0.994917  0.003324   
Log loss                          0.249982  0.245637  0.409774  0.097156   
Fit time (s)                      2.052726  0.008036  0.124394  0.318551   
Predict time (s)                  0.017792  0.000812  0.004608  0.001021   

                                                      
Estimator                           logreg        rf  
Metric           Label / Average                      
Accuracy                          0.033581  0.029019  
Precision        0                0.000000  0.015232  
                 1                0.093552  0.018444  
                 2                0.034702  0.043005  
                 3                0.040507  0.044810  
                 4                0.028518  0.028181  
                 5                0.031803  0.041755  
                 6                0.039446  0.033112  
                 7                0.043151  0.073190  
                 8                0.103881  0.096057  
                 9                0.127703  0.121112  
Recall           0                0.023860  0.015434  
                 1                0.049183  0.023340  
                 2                0.101130  0.081546  
                 3                0.180135  0.130635  
                 4                0.027786  0.036218  
                 5                0.040751  0.054250  
                 6                0.023241  0.030429  
                 7                0.139443  0.109361  
               

## Stability vs mean

A single mean accuracy hides how noisy a model is across folds. Two estimators with near-identical means can differ by a factor of two or more in per-fold standard deviation, and for production use that spread matters as much as the headline score. The next cell pulls the raw per-fold accuracies, ranks by mean, and then calls out the case where the top-mean model is not the tightest across folds.

In [6]:
def per_fold_accuracy(report):
    """Return the per-fold accuracy values from a CrossValidationReport."""
    frame = report.metrics.accuracy(aggregate=None)
    return frame.iloc[0].astype(float).to_numpy()


stats = {}
for slug, report in reports.items():
    folds = per_fold_accuracy(report)
    stats[slug] = (float(folds.mean()), float(folds.std(ddof=1)))

ranked = sorted(stats.items(), key=lambda kv: kv[1][0], reverse=True)

print(f"=== Ranking by mean {METRIC} (5-fold CV) ===")
print(f"{'rank':<5}{'model':<10}{'mean':>10}{'std':>10}")
for i, (slug, (mean, std)) in enumerate(ranked, start=1):
    print(f"{i:<5}{slug:<10}{mean:>10.4f}{std:>10.4f}")

top_slug, (top_mean, top_std) = ranked[0]
most_stable_slug, (stable_mean, stable_std) = min(
    stats.items(), key=lambda kv: kv[1][1]
)
if most_stable_slug != top_slug and stable_std > 0:
    ratio = top_std / stable_std
    print(
        f"\nNote: {top_slug} has the top mean {METRIC.lower()} "
        f"({top_mean:.4f}) but {most_stable_slug} has ~{ratio:.1f}x "
        f"tighter per-fold spread (std {stable_std:.4f} vs {top_std:.4f})."
    )
else:
    print(
        f"\nNote: {top_slug} leads on both mean ({top_mean:.4f}) "
        f"and per-fold stability (std {top_std:.4f}) for this run."
    )

Output()

Output()

Output()

=== Ranking by mean Accuracy (5-fold CV) ===
rank model           mean       std
1    rf            0.9377    0.0290
2    hgb           0.9349    0.0185
3    logreg        0.9204    0.0336

Note: rf has the top mean accuracy (0.9377) but hgb has ~1.6x tighter per-fold spread (std 0.0185 vs 0.0290).


## Verify in MLflow

Finally, we ask `project.summarize()` for the MLflow-side view: three rows, one per run, each tagged with the `key` we used when calling `project.put(...)`. The same rows show up in the MLflow UI (`uv run mlflow ui --backend-store-uri sqlite:///mlflow.db`), where you can drill into per-fold metrics and download the pickled report.

In [7]:
project.summarize()[["key", "report_type", "learner", "ml_task", "dataset"]]

ValueError: Dataframe is missing required columns: ['ml_task', 'dataset', 'learner', 'fit_time', 'predict_time', 'rmse', 'log_loss', 'roc_auc', 'fit_time_mean', 'predict_time_mean', 'rmse_mean', 'log_loss_mean', 'roc_auc_mean']

key       report_type  \
  id                                                           
0 8b1f991029c1471cb31ea5183724cb65     hgb  cross-validation   
1 4c3274458f8e4f3bbab15987f7f65dda  logreg  cross-validation   
2 d3c513061617460282a156b6b8c2d681      rf  cross-validation   

                                                           learner  \
  id                                                                 
0 8b1f991029c1471cb31ea5183724cb65  HistGradientBoostingClassifier   
1 4c3274458f8e4f3bbab15987f7f65dda              LogisticRegression   
2 d3c513061617460282a156b6b8c2d681          RandomForestClassifier   

                                                      ml_task   dataset  
  id                                                                     
0 8b1f991029c1471cb31ea5183724cb65  multiclass-classification  f7498164  
1 4c3274458f8e4f3bbab15987f7f65dda  multiclass-classification  f7498164  
2 d3c513061617460282a156b6b8c2d681  multiclass-classification  f7498164